# Reconstructing the whole Arctic from sparse observations

So far the network has only ever been asked about places where an ITP was. But its inputs — wind, ice concentration,
date, position — are known *everywhere*. Nothing stops us from feeding it the inputs of every grid cell of the Arctic:

```{figure} ../figures/sparse_to_global.png
:width: 100%
:name: sparse-to-global

Train at the observation locations; then give the same network the inputs on the whole grid, and compare the predicted
ice velocity with the model's truth. (The two single-ITP networks of the [landing page](intro.md) are the same picture with less training data.)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

We train the network of the previous page again (the pages are independent notebooks), then apply it to the ten days
of full-Arctic maps that come with the dataset. `ml.reconstruct` does the bookkeeping: the same seven inputs, the same
scaling, the same network, just on 65 × 360 grid cells instead of along tracks.

In [ ]:
p = data.prepared()
model, history = ml.train(p, hp=ml.HP)

Y_rec = ml.reconstruct(model, p.scaler)        # (y, x, day, [u, v]) in m/s, NaN over land
print("reconstruction:", Y_rec.shape, "   truth:", data.maps.Y_true.shape)

## How does it look?

One day: the true ice speed from the model, the reconstruction, and the magnitude of the vector difference.
Only ice-covered ocean cells (mean concentration above 15 %) are shown and scored — a reconstruction of
ice velocity in open water means nothing.

In [ ]:
plots.reconstruction_day(Y_rec, t=0)

## How good is it?

**Pooled** over all ice-covered cells and all ten days, the reconstruction is scored exactly like the test set on the
previous page:

In [ ]:
metrics.pooled_scores(Y_rec);

The skill is noticeably lower than on the test members (about 0.56 against 0.70). That gap is the whole point of the
exercise: the test members are scored *along the ITP tracks*, where the network has seen the inputs in training;
the reconstruction asks it about the Siberian shelves, the Fram Strait outflow and the Canadian Archipelago, which
no ITP ever visited. The network has to *extrapolate* there.

Where exactly it extrapolates well or badly is shown by scoring each grid cell separately over the ten days:

In [ ]:
s_map, c_map = metrics.cell_scores(Y_rec)
plots.cell_scores(s_map, c_map, title="per-cell skill and correlation of the reconstruction, 10 days")

```{warning}
Ten days is a short record for per-cell statistics, so these maps are noisy (and cells where the true velocity barely
varied over the ten days are masked). Read the patterns, not individual cells. The pooled skill above is the robust number.
```

The pattern is what you would expect from the track map on the [data page](01_data.ipynb): the Beaufort Gyre and the central
Arctic, where the ITPs were, score well; the coasts of Siberia and the Canadian Archipelago — where the ice is
land-fast or strongly sheared, and where there were no observations — score poorly or even negatively.

So the skill of a reconstruction depends on **where the observations were**. Turn that around and it becomes a design
question: *given that we can only afford a few instruments, where should they go?*

Next: [Observing System Design](06_osd.ipynb).